# 第9回 主成分分析を実装する


---


## 前回からの接続


第1回から第8回までは、いつも正解のラベルがありました。利用台数、価格、心臓病、ペンギンの種。ラベルを当てるのが目的でした。

今日から、ラベルを使わない分析に入ります。最初に扱うのは、たくさんの列を持つデータを少ない本数に縮める方法、主成分分析（principal component analysis）（PCA）です。題材は顔の写真。1枚の顔は、4096個の数でできています。


---


## 今日の分析目標


あなたは顔認証システムの開発チームの一員です。400枚の顔写真を、見分ける力を落とさずに、できるだけ少ない数で持ちたい。1枚あたり何個の数で持てばよいか。

顔の散らばりの「向き」を行列の計算で求め、少ない向きで顔を組み立て直し、見分ける力が落ちないかを確かめます。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは顔認証システムの開発チームの一員です。システムは、人ごとに顔写真をためておきます。1枚の顔写真は、たくさんの数でできています。

決めたいのは、顔写真をできるだけ少ない数で持つことです。でも、人を見分ける力は落としたくありません。


### 手元のデータ

Olivettiの顔写真です。scikit-learnの`fetch_olivetti_faces`で読み込みます。40人が10枚ずつ、合わせて400枚で、1行が1枚の顔です。

答えの列のかわりに、人物の番号（0〜39）が付いています。縮めるときには使わず、見分ける力を確かめるときだけ使います。


### 最後に出す答え

1枚あたり何個の数で持てばよいかと、そのとき人を見分ける正解率がどうなるかです。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 表を行列として見て、転置（`.T`）と行列の積（`@`）で共分散行列（covariance matrix）を作れる
- `np.linalg.eigh`で固有値分解（eigendecomposition）し、scikit-learnの`PCA`と同じ答えになることを確かめられる
- 主成分（固有顔）と累積寄与率（cumulative proportion of variance explained）から、何本で何割の情報を保てるかを読める
- 平均の顔に固有顔を足して、顔を組み立て直せる
- 次元を減らしても、人物を見分ける力が落ちないかを交差検証（cross-validation）で確かめられる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ② の解析（縮める）と、③ の解釈（何個で持つか）が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. 顔を、数の表として見る


少ない数で持つ前に、いまの顔が何個の数でできているかを数えておきます。Olivettiの顔写真です。40人が10枚ずつ、合わせて400枚。1枚は縦64×横64の白黒で、1つのマス（画素）が0（黒）〜1（白）の明るさの数です。


### おさらい：numpyの配列

```python
x = df['temp'].values    # 列を配列に
x[0]                     # 0番目を取り出す
```

配列は数の並びで、番号で取り出せ、まとめて計算できるのでした（第2回）。`.shape`で、縦と横の大きさがわかります。


### おさらい：def

```python
def 名前(引数):
    return 戻り値
```

`def`の後ろが名前と引数（入力）、`return`の後ろが戻り値（出力）でした（第3回）。このあと書く`show`のように、`return`がなく、描くだけの関数もあります。


### お手本：読み込んで、大きさを見る

- 役割：顔写真を読み込み、表の大きさを確かめ、1枚を画像に戻して表示する
- 入力：なし（インターネットから読み込む）
- 出力：表`X`（400行×4096列）、人物の番号`y`、1枚の顔の図

`X`の1行が1枚の顔で、4096個の明るさが横に並んでいます。`reshape(64, 64)`で、横一列の4096個を、縦64×横64のマス目に並べ直すと、画像に戻ります。表示の手順は、何度も使うので関数`show`にしておきます。

`plt.subplots(figsize=…)`は、図（`fig`）と、絵を描く枠（`ax`）を1つ作ります。`show(ax, 顔, 題)`と呼ぶと、その枠に顔を描きます。


In [ ]:
faces = fetch_olivetti_faces(shuffle=False)
X, y = faces.data, faces.target      # 顔の表と、人物の番号（0〜39）
print(X.shape)

def show(ax, v, title):              # 4096個の並びを、顔の画像で表示
    ax.imshow(v.reshape(64, 64), cmap='gray')
    ax.set_title(title); ax.axis('off')

fig, ax = plt.subplots(figsize=(2.5, 2.5))
show(ax, X[0], '0枚目の顔')
plt.show()


`(400, 4096)`。400行（枚）× 4096列（画素）の表です。400枚をそのまま持つと、400 × 4096 ＝ 1,638,400個の数になります。

でも、顔どうしは似ています。目・鼻・口の位置はだいたい同じです。4096個の数は、ばらばらに動いているわけではなさそうです。顔どうしの違いが大きい「向き」だけを取り出せば、少ない数で持てるかもしれません。


---


## 2. 散らばりの向きを、行列で求める


では、顔どうしの違いが大きい向きは、どう求めればよいでしょうか。この向きは、共分散行列の固有ベクトル（eigenvector）として求まります。PCAの中身は、この計算です。

- まず、各画素から平均を引く（中心化）。これで「平均の顔からのずれ」の表になる
- 次に、画素どうしが一緒に動く度合いを並べた共分散行列 $S = X_c^\top X_c / (n - 1)$ を作る（4096×4096）
- $S$ を固有値分解すると、$S v = \lambda v$ を満たす向き $v$（固有ベクトル）と、その向きの散らばりの大きさ $\lambda$（固有値）が出てくる


### お手本：numpyで、固有値分解する

- 役割：中心化（centering）した表から共分散行列を作り、固有値分解して、散らばりの大きい順に並べる
- 入力：顔の表`X`
- 出力：固有値（eigenvalue）`vals`（大きい順）と、固有ベクトル`vecs`

`.T`は転置（行と列を入れかえる）、`@`は行列の積です。`Xc`は400×4096、`Xc.T`は4096×400なので、`Xc.T @ Xc`は4096×4096になります。`np.linalg.eigh`は、共分散行列のような対称な行列の固有値分解をする関数で、固有値を小さい順に返します。`[::-1]`で大きい順に並べかえます。計算に10秒ほどかかります。


In [ ]:
Xc = X - X.mean(axis=0)              # 中心化：平均の顔からのずれ
S = Xc.T @ Xc / (len(X) - 1)         # 共分散行列（4096×4096）
print(Xc.shape, Xc.T.shape, S.shape)
vals, vecs = np.linalg.eigh(S)       # 固有値分解（小さい順）
vals, vecs = vals[::-1], vecs[:, ::-1]   # 大きい順に並べかえる
print(vals[:3].round(3))


大きい順に18.840、11.072、6.305と、急に小さくなっていきます。`vecs`の列が固有ベクトルで、`vecs[:, 0]`が1番目の向き（4096個の数）です。


### 変えてみる：scikit-learnのPCAと比べる

同じ計算を、`PCA`で行います。`explained_variance_`が固有値、`components_`の行が固有ベクトル（主成分）です。


In [ ]:
pca = PCA().fit(X)
print(pca.explained_variance_[:3].round(3))
print(np.allclose(np.abs(vecs[:, 0]), np.abs(pca.components_[0]),
                  atol=1e-4))


固有値は18.840、11.072、6.305で、自分で計算したものと一致します。1本目の固有ベクトルも`True`（ここで比べたのは1本目だけです）。`PCA`の中身は、今やった固有値分解そのものです。

`np.abs`で絶対値をとって比べたのは、固有ベクトルは向きが逆でも同じ答えだからです（$v$ も $-v$ も $S v = \lambda v$ を満たす）。PCAの主成分（principal component）の符号は、計算の都合で入れかわることがあります。


---


## 3. 固有顔と、累積寄与率


向きは求まりました。では、その向きはどんな形をしているでしょうか。主成分は4096個の数の並びなので、`reshape(64, 64)`すれば画像として見られます。これを固有顔と呼びます。


### お手本：平均の顔と、固有顔を並べる

- 役割：平均の顔と、上位9個の主成分を画像で並べる
- 入力：`X`、`pca`
- 出力：10枚の図

`plt.subplots(2, 5)`は、2行5列、10個の枠を作ります。`axes`は枠の2次元の配列で、`axes[0, 0]`は0行0列、つまり左上の枠です。`axes.ravel()`は、2行5列の枠を1列の10個に並べ直し、`[1:]`で左上を除いた9個にします。

`enumerate(並び)`は、並びの中身を番号といっしょに取り出します。`i`が0, 1, 2, …、`ax`が枠で、`i`番目の主成分を「固有顔i＋1」として描きます。


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.4))
show(axes[0, 0], X.mean(axis=0), '平均の顔')
for i, ax in enumerate(axes.ravel()[1:]):
    show(ax, pca.components_[i], f'固有顔{i + 1}')
plt.tight_layout(); plt.show()


固有顔1は、顔全体の明るさ（照明の当たり方）のような形をしています。固有顔2は左右から、固有顔3は上下からの光の当たり方のような形です。上位の固有顔は、人の違いより先に、照明の違いを拾っているようです。後ろの固有顔ほど、目元や口元など、細かい部分の違いが現れます。

どの顔も、「平均の顔 ＋ 固有顔1 × いくつ ＋ 固有顔2 × いくつ ＋ …」の形で書けます。この「いくつ」が、その顔の主成分得点です。


各主成分が、全体の散らばりの何割を受け持つかを寄与率（proportion of variance explained）と呼びます。固有値を、固有値の合計で割ったものです。上位から足していった累積寄与率を見れば、何個の主成分で何割の情報を保てるかがわかります。


### TODO①：上位50個で、何割を保てるか

`pca.explained_variance_ratio_`（寄与率）を`np.cumsum`で累積して、上位50個の主成分で全体の何割を保てるかを表示してください。5・10・20・100個でも表示してみましょう。


In [ ]:
# TODO: 累積寄与率を計算し、上位50個で保てる割合を表示する
# ヒント: cum = np.cumsum(pca.explained_variance_ratio_)
#         上位50個は cum[49]（番号は0から）
...


上位50個で87.4%。4096個のうち、たった50個の向きで、顔の散らばりの9割近くを保てます。5個で54.4%、10個で65.6%、20個で76.3%、100個で93.5%。はじめは急に増え、だんだん頭打ちになります。


In [ ]:
# @title 図：累積寄与率
cum = np.cumsum(pca.explained_variance_ratio_)
plt.plot(np.arange(1, 201), cum[:200], lw=2.5, color='#0066cc')
plt.axhline(0.9, color='#e63946', ls='--', lw=1.5)
plt.xlabel('主成分の数'); plt.ylabel('累積寄与率')
plt.show()


---


## 4. 顔を組み立て直す


上位50個で、散らばりの9割近くを保てました。では、顔はどこまで元に戻るでしょうか。平均の顔に、固有顔を少しずつ足して、1枚の顔を組み立て直します（再構成）。

$$
\text{再構成} = \text{平均の顔} + \sum_{j=1}^{k} (\text{得点}_j) \times (\text{固有顔}_j)
$$


### お手本：0枚目の顔を、5〜100個で組み立て直す

- 役割：0枚目の顔の得点を上位k個の固有顔で求め、平均の顔に足し戻す
- 入力：`X[0]`、`pca.components_`
- 出力：元の顔と、4通りの再構成

`plt.subplots(1, 5)`のように1行だけのときは、`axes`は1列の配列になり、`axes[0]`が左端の枠です。`zip(axes[1:], [5, 20, 50, 100])`は、左端を除いた4つの枠と4つの個数を、同じ順に1組ずつ取り出します。

`V = pca.components_[:k]`はk×4096の行列です。`face @ V.T`で、顔のずれをk個の固有顔それぞれに当てた得点（k個）が出ます。`得点 @ V`で、得点 × 固有顔を足し合わせた顔（4096個）に戻ります。


In [ ]:
face = X[0] - X.mean(axis=0)          # 平均の顔からのずれ
fig, axes = plt.subplots(1, 5, figsize=(11, 2.6))
show(axes[0], X[0], '元の顔')
for ax, k in zip(axes[1:], [5, 20, 50, 100]):
    V = pca.components_[:k]             # 上位 k 個の固有顔（k×4096）
    rec = X.mean(axis=0) + (face @ V.T) @ V
    show(ax, rec, f'{k}個')
plt.tight_layout(); plt.show()


5個では、ぼんやりした「だれかの顔」です。20個で輪郭と明るさがそろい、50個で本人とわかる顔になります。100個でも、元の顔にはない眼鏡の影が少し残ります。ほかの人の眼鏡が混ざって見えているのです。4096個の数を、50個の得点で持てば、顔として十分に伝わります。

ただし、ここで組み立て直したのは、PCAを学んだ400枚そのものです。学んでいない新しい人の顔なら、同じ個数でも、もっと崩れるはずです。


### TODO②：別の顔を、10個と200個で組み立て直す

100枚目の顔`X[100]`を選び、10個と200個で組み立て直して、元の顔と並べて表示してください。


In [ ]:
# TODO: X[100] を 10個と200個で組み立て直し、元の顔と並べる
# ヒント: お手本の X[0] を X[100] に、
#         [5, 20, 50, 100] を [10, 200] に。
#         plt.subplots(1, 3, …) にする
...


10個では輪郭と明るさだけ、200個では細かいしわまで元に近づきます。成分を増やすほど元の顔に近づき、そのぶん持つ数も増えます。


---


## 5. 縮めても、人を見分けられるか


少ない数で持てても、顔認証で人を見分けられなければ意味がありません。ここだけ、人物の番号`y`を使って確かめます。


### お手本：4096個のままと、50個に縮めたもので比べる

- 役割：ロジスティック回帰（logistic regression）で、人物（40人）を当てる正解率を、5分割交差検証で比べる
- 入力：`X`、`y`
- 出力：2つの正解率

縮めるほうは、`PCA(n_components=50)`とロジスティック回帰をパイプライン（pipeline）につなぎます（第7回）。PCAの向きも、交差検証の訓練の中だけで学ばれます。計算に30秒ほどかかります。


In [ ]:
full = cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=5)
pipe = Pipeline([('pca', PCA(n_components=50, random_state=0)),
                 ('model', LogisticRegression(max_iter=2000))])
red = cross_val_score(pipe, X, y, cv=5)
print('4096個:', round(full.mean(), 3))
print('  50個:', round(red.mean(), 3))


どちらも0.972。1枚あたりの数を約80分の1（4096個→50個）に縮めても、人を見分ける力は落ちませんでした。このデータでは、上位50個の向きに、見分けに要る情報もほぼ含まれていたことになります。

ただし、いつもそうとは限りません。3節で見たとおり、上位の向きは照明の違いを拾いやすく、大きく散らばる向きが、そのまま人を見分ける手がかりだとは言えません。縮めたら、見分ける力を今のように確かめる必要があります。


---


## 6. 答えを出す：何個の数で持つか


縮めても、見分ける力は落ちませんでした。では、持つ数は何分の1になるでしょうか。最初の問いに戻ります。50個で持つとき、実際に覚えておく数は何個でしょうか。要るのは、平均の顔（4096個）、固有顔50枚（4096 × 50個）、各顔の得点（400 × 50個）の3つです。


### お手本：50個で持つときの、数の個数

- 役割：そのまま持つときと、50個で持つときの、数の個数を比べる
- 入力：表の大きさ、k＝50
- 出力：2つの個数と、その比


In [ ]:
n, d = X.shape
k = 50
full_size = n * d                    # そのまま：400枚 × 4096
pca_size = d + d * k + n * k         # 平均の顔 ＋ 固有顔 k 枚 ＋ 得点
print(f'そのまま : {full_size:,} 個')
print(f'{k}個で持つ: {pca_size:,} 個（{pca_size / full_size:.1%}）')


そのままなら1,638,400個、50個で持てば228,896個。およそ7分の1（14.0%）です。


### TODO③：9割を保つには、何個か

累積寄与率が初めて0.9以上になる主成分の数を求めてください。


In [ ]:
# TODO: 累積寄与率が初めて 0.9 以上になる主成分の数を求める
# ヒント: cum = np.cumsum(pca.explained_variance_ratio_)
#         np.argmax(cum >= 0.9) は、最初に True になる番号（0から）
...


66個です。`cum >= 0.9`はTrue/Falseの並びで（第6回）、`np.argmax`は最初のTrueの番号を返します。番号は0から数えるので、1を足して66個です。


### TODO④：個数と、保てる割合の表

kを20・50・66・100と変えて、数の個数、そのままとの比、累積寄与率を1行ずつ表示してください。


In [ ]:
# TODO: k = 20, 50, 66, 100 で、個数・比・累積寄与率を表示する
# ヒント: for k in [20, 50, 66, 100]: の中で、
#         お手本の pca_size を計算し、cum[k - 1] も表示する
#         （cum は TODO③のヒントと同じ）
...


20個で5.7%・76.3%、50個で14.0%・87.4%、66個で18.4%・90.0%、100個で27.7%・93.5%。持つ数を増やしても、保てる割合の伸びは小さくなっていきます。

> 答え：1枚あたり50個の数（得点）で持てばよい。平均の顔と固有顔50枚を合わせても、数は7分の1になり、散らばりの87.4%を保てる。人を見分ける正解率は0.972で、4096個のままと変わらない。

まだ言えないことが、2つ残っています。

- 軸の意味：固有顔は絵として眺められたが、表のデータの主成分は絵にならない。第1主成分は何を表すのか
- ちょうどよい個数：50個より多くしたら、正解率はもっと上がるのか（課題）


---


## 目標に答えられたか


- 今日の目標は「400枚の顔写真を、見分ける力を落とさずに、できるだけ少ない数で持つ」ことでした。6節の答えで、数は何分の1になり、正解率はどうでしたか？
- 2節で、`Xc.T @ Xc`の大きさはいくつでしたか？ なぜその大きさになりましたか？
- 3節の固有顔は、どんな「顔の違い」を表していましたか？
- TODO①で、上位50個の主成分は全体の何割を保てましたか？


---


## 今日の要点


- PCAは、データの散らばりが大きい向きを、大きい順に取り出す。中身は共分散行列の固有値分解
- 固有値は、その向きの散らばりの大きさ。寄与率は、固有値 ÷ 固有値の合計
- 主成分を画像に戻したものが固有顔。どの顔も「平均の顔 ＋ 得点 × 固有顔の和」で書ける
- 累積寄与率で、何個で何割を保てるかを読む。はじめ急に増え、だんだん頭打ち
- 顔は50個で、数は7分の1、正解率は変わらない
- Python：`.T`で転置、`@`で行列の積、`np.linalg.eigh`で固有値分解


---


## 次回へ


今日の主成分は、顔の画像として眺められました。でも、表のデータ（車の性能表など）の主成分は、絵になりません。

次回は、車の性能表で、主成分の意味を読みます。主成分は、どの列と関係が強いのでしょうか。2本の軸に名前をつけて、地図の上の位置で1台の車を説明できるでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

5節では、主成分の数を50に決めて人物を当てました。今度は、主成分の数を10・50・100の3通りに変えて、同じ手順（`PCA` → `LogisticRegression`のパイプラインを5分割交差検証）で、それぞれの正解率を表示してください。結果は、辞書`accs`に`{主成分の数: 正解率}`の形で集めておきましょう。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

5節のお手本の`n_components=50`を変えるだけです。
`accs = {}`を作り、`for k in [10, 50, 100]:`の中で`accs[k] = cross_val_score(…).mean()`とします。

</details>


### 応用②（判断）

応用①の3つのうち、正解率が初めて0.95を超える主成分の数はどれですか。10・50・100の整数で答えてください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

3つの値を小さい順に見て、最初に0.95を超えたものを答えます。

</details>


### 応用③（解釈）

主成分の数を増やせば、正解率は必ず上がるのでしょうか。応用①の3つの正解率と、TODO①の累積寄与率（10・50・100個）を根拠に、開発チームに向けて「何個で持つのがよいか」を3行程度で書いてください。


（ここに3行程度で書く）


---


## 発展（任意）


### 固有顔の向きに、顔を動かす

固有顔は「平均の顔から、どの向きにずれるか」でした。平均の顔に、固有顔を足したり引いたりすると、その向きに顔が動く様子が見えます。

動かす量は、その向きの散らばりの大きさ（標準偏差 $\sqrt{\lambda}$）を単位にします。−2から ＋2まで動かしてみます。

ここでは`show`を使わず、`imshow`に`vmin=0, vmax=1`を渡します。`imshow`は、ふつう画像ごとに、いちばん暗い所を黒、いちばん明るい所を白に伸ばして表示します。それだと、顔全体が明るくなっても、見た目がほとんど変わりません。`vmin=0, vmax=1`で、明るさの目盛りを、どの画像でも0（黒）〜1（白）に固定します。


In [ ]:
mean_face = X.mean(axis=0)
steps = [-2, -1, 0, 1, 2]
fig, axes = plt.subplots(3, 5, figsize=(10, 6.5))
for r in range(3):                                   # 固有顔1〜3
    sd = np.sqrt(pca.explained_variance_[r])         # その向きの標準偏差
    for c, s in enumerate(steps):
        v = mean_face + s * sd * pca.components_[r]
        ax = axes[r, c]
        ax.imshow(v.reshape(64, 64), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'固有顔{r + 1} {s:+d}'); ax.axis('off')
plt.tight_layout(); plt.show()


読み方　真ん中の列（0）は、どれも平均の顔です。1行目（固有顔1）は、左右に動かすと顔全体の明るさが変わります（目盛りを固定したので、明るさの違いが見えます）。2行目では左右から、3行目では上下からの光の当たり方が変わります。

主成分の得点は、「この向きに、平均からどれだけずれているか」を表す数だったのです。固有顔の符号は計算の都合で決まるので、＋と−のどちらが明るいかには意味がありません。

試すなら、`range(3)`を`range(3, 6)`に変えて、固有顔4〜6の向きも動かしてみましょう。後ろの固有顔ほど、目元や口元の細かい動きになります。
